# Tạo sẵn embedding query VI / EN / ZH

Chọn **Runtime → Change runtime type → T4 GPU**, sau đó **Run all** và tải lên
file Parquet có `id`, `query`, `english`, `chinese`.

Output giữ nguyên mọi cột và thứ tự đầu vào, thêm `vector_original`,
`vector_english`, `vector_chinese` (mỗi vector 768 số float32, chuẩn hóa L2),
và `query_sha256` (hash ba chuỗi query). Metadata encoder được lưu ngay trong
schema Parquet; không cần file manifest riêng. Không in nội dung query ra output.

Encoder và xử lý cửa sổ khớp `BackEnd/app/pipeline.py` và
`Scripts/Qdrant/build_gte_qdrant.py`: CLS, tối đa 8192 token/cửa sổ,
chuẩn hóa từng cửa sổ rồi lấy trung bình theo số token và chuẩn hóa lại.
Giữ text cho BM25 và reranker. Chỉ dùng các vector này với index GTE tương thích.

Model công khai: https://huggingface.co/Alibaba-NLP/gte-multilingual-base


In [ ]:
# Chỉ cài trong runtime Colab; giữ nguyên PyTorch/CUDA do Colab cung cấp.
import subprocess
import sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.51.3', 'einops==0.8.1', 'pyarrow==19.0.1',
])


In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import os
import tempfile
from time import perf_counter

import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import torch
import transformers
from transformers import AutoConfig, AutoModel, AutoTokenizer

# Để trống INPUT_PATH để chọn file từ máy. Có thể điền đường dẫn /content/...
INPUT_PATH = ''
OUTPUT_PATH = ''  # Mặc định: <tên đầu vào>_embeddings.parquet
ROW_BATCH_SIZE = 32
ENCODER_BATCH_SIZE = 8
TOKEN_BUDGET = 8192
REQUIRE_GPU = True
DOWNLOAD_OUTPUT = True
OVERWRITE_OUTPUT = False

ENCODER = {
    'model': 'Alibaba-NLP/gte-multilingual-base',
    'revision': '9bbca17d9273fd0d03d5725c7a4b0f6b45142062',
    'code_revision': '40ced75c3017eb27626c9d4ea981bde21a2662f4',
    'dimension': 768,
    'max_length': 8192,
    'pooling': 'CLS',
    'window_aggregation': 'token-weighted mean of L2-normalized windows, then L2',
    'normalized': True,
    'storage_dtype': 'float32',
    'query_prefix': '',
}
VECTOR_COLUMNS = {
    'query': 'vector_original',
    'english': 'vector_english',
    'chinese': 'vector_chinese',
}


In [ ]:
if not INPUT_PATH:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Chọn đúng một file Parquet.')
    INPUT_PATH = next(iter(uploaded))
    del uploaded

input_path = Path(INPUT_PATH).resolve()
output_path = (Path(OUTPUT_PATH).resolve() if OUTPUT_PATH else
               input_path.with_name(input_path.stem + '_embeddings.parquet'))
if output_path == input_path:
    raise ValueError('Output phải khác file đầu vào.')
if output_path.exists() and not OVERWRITE_OUTPUT:
    raise FileExistsError(f'{output_path.name} đã tồn tại; đổi OUTPUT_PATH hoặc bật OVERWRITE_OUTPUT.')
if min(ROW_BATCH_SIZE, ENCODER_BATCH_SIZE, TOKEN_BUDGET) < 1:
    raise ValueError('Batch size và token budget phải dương.')

parquet = pq.ParquetFile(input_path)
required = ['id', *VECTOR_COLUMNS]
if len(set(parquet.schema_arrow.names)) != len(parquet.schema_arrow.names):
    raise ValueError('Input có tên cột trùng nhau.')
if not set(required) <= set(parquet.schema_arrow.names):
    raise ValueError(f'Input cần các cột {required}.')
if set([*VECTOR_COLUMNS.values(), 'query_sha256']) & set(parquet.schema_arrow.names):
    raise ValueError('Input đã có cột embedding/hash; hãy dùng file query gốc.')
if parquet.metadata.num_rows == 0:
    raise ValueError('Input không có query.')

seen_ids = set()
for batch in parquet.iter_batches(batch_size=ROW_BATCH_SIZE, columns=required):
    for row in batch.to_pylist():
        identifier = row['id']
        if not isinstance(identifier, (int, str)) or isinstance(identifier, bool):
            raise ValueError('ID phải là số nguyên hoặc chuỗi không null.')
        if identifier in seen_ids:
            raise ValueError('Input có ID trùng nhau.')
        seen_ids.add(identifier)
        if any(not isinstance(row[name], str) or not row[name].strip() for name in VECTOR_COLUMNS):
            raise ValueError('Cả ba query phải là chuỗi không rỗng.')
del seen_ids

digest = hashlib.sha256()
with input_path.open('rb') as stream:
    for block in iter(lambda: stream.read(8 * 1024**2), b''):
        digest.update(block)
source_sha256 = digest.hexdigest()
print(f'Input hợp lệ: {parquet.metadata.num_rows:,} dòng, {parquet.metadata.num_rows * 3:,} vector.')


In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if REQUIRE_GPU and device.type != 'cuda':
    raise RuntimeError('Hãy chọn Runtime → Change runtime type → T4 GPU và chạy lại.')
inference_dtype = torch.float16 if device.type == 'cuda' else torch.float32
options = {'revision': ENCODER['revision']}
tokenizer = AutoTokenizer.from_pretrained(ENCODER['model'], **options)
model_config = AutoConfig.from_pretrained(
    ENCODER['model'], trust_remote_code=True,
    code_revision=ENCODER['code_revision'], **options,
)
model = AutoModel.from_pretrained(
    ENCODER['model'], config=model_config, trust_remote_code=True,
    code_revision=ENCODER['code_revision'], torch_dtype=inference_dtype,
    attn_implementation='sdpa', low_cpu_mem_usage=False, _fast_init=False,
    **options,
)
# Các buffer RoPE/position không nằm trong checkpoint GTE.
with torch.device('cpu'):
    model.embeddings.register_buffer(
        'position_ids', torch.arange(model_config.max_position_embeddings), persistent=False,
    )
    model.embeddings._init_rope(model_config)
if model_config.hidden_size != ENCODER['dimension']:
    raise ValueError('Số chiều model khác cấu hình encoder.')
model.eval().to(device)
print('Device:', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU')
print('Model:', ENCODER['model'], '| inference:', str(inference_dtype))


In [ ]:
def normalize_vectors(vectors):
    vectors = np.ascontiguousarray(vectors, dtype=np.float32)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    if not np.isfinite(vectors).all() or np.any(norms <= 1e-12):
        raise ValueError('Embedding chứa NaN/Inf hoặc vector zero.')
    return vectors / norms


def forward(windows):
    try:
        features = tokenizer.pad(windows, padding=True, return_tensors='pt')
        features = {name: tensor.to(device) for name, tensor in features.items()}
        with torch.inference_mode():
            vectors = model(**features).last_hidden_state[:, 0].float().cpu().numpy()
        return normalize_vectors(vectors)
    except torch.cuda.OutOfMemoryError:
        if len(windows) == 1:
            raise RuntimeError('GPU thiếu VRAM cho một cửa sổ; không bỏ qua query.') from None
    gc.collect()
    torch.cuda.empty_cache()
    middle = len(windows) // 2
    return np.concatenate([forward(windows[:middle]), forward(windows[middle:])])


def encode_texts(texts):
    tokens = tokenizer(
        texts, add_special_tokens=False, truncation=False,
        return_attention_mask=False, return_token_type_ids=False,
    )['input_ids']
    payload = ENCODER['max_length'] - tokenizer.num_special_tokens_to_add(pair=False)
    windows, owners, weights = [], [], []
    for owner, ids in enumerate(tokens):
        for start in range(0, max(1, len(ids)), payload):
            piece = ids[start:start + payload]
            windows.append({'input_ids': tokenizer.build_inputs_with_special_tokens(piece)})
            owners.append(owner)
            weights.append(max(1, len(piece)))
    lengths = [len(window['input_ids']) for window in windows]
    order = sorted(range(len(windows)), key=lengths.__getitem__, reverse=True)
    sums = np.zeros((len(texts), ENCODER['dimension']), dtype=np.float32)
    offset = 0
    while offset < len(order):
        size = min(ENCODER_BATCH_SIZE, max(1, TOKEN_BUDGET // lengths[order[offset]]))
        positions = order[offset:offset + size]
        vectors = forward([windows[position] for position in positions])
        for position, vector in zip(positions, vectors, strict=True):
            sums[owners[position]] += vector * weights[position]
        offset += size
    return normalize_vectors(sums)


def query_hash(row):
    content = json.dumps([row[name] for name in VECTOR_COLUMNS], ensure_ascii=False, separators=(',', ':'))
    return hashlib.sha256(content.encode('utf-8')).hexdigest()


def validate_output(path):
    result = pq.ParquetFile(path)
    metadata = json.loads(result.schema_arrow.metadata[b'query_embedding'])
    if metadata['source_sha256'] != source_sha256 or metadata['encoder'] != ENCODER:
        raise ValueError('Metadata encoder/source không khớp.')
    if result.metadata.num_rows != parquet.metadata.num_rows:
        raise ValueError('Output thiếu/thừa dòng.')
    vector_type = pa.list_(pa.float32(), ENCODER['dimension'])
    for name in VECTOR_COLUMNS.values():
        if result.schema_arrow.field(name).type != vector_type:
            raise ValueError('Sai kiểu hoặc số chiều vector.')
    for original, embedded in zip(
        parquet.iter_batches(batch_size=ROW_BATCH_SIZE),
        result.iter_batches(batch_size=ROW_BATCH_SIZE), strict=True,
    ):
        if not pa.Table.from_batches([embedded]).select(parquet.schema_arrow.names).equals(
            pa.Table.from_batches([original])
        ):
            raise ValueError('Output thay đổi dữ liệu/thứ tự đầu vào.')
        expected_hashes = [query_hash(row) for row in original.to_pylist()]
        if embedded.column('query_sha256').to_pylist() != expected_hashes:
            raise ValueError('Query hash không khớp.')
        for name in VECTOR_COLUMNS.values():
            values = embedded.column(name)
            vectors = np.asarray(values.to_pylist(), dtype=np.float32)
            if values.null_count or not np.isfinite(vectors).all():
                raise ValueError('Vector bị null/NaN/Inf.')
            if not np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-5):
                raise ValueError('Vector chưa chuẩn hóa L2.')
    return result.metadata.num_rows


In [ ]:
started = perf_counter()
metadata = dict(parquet.schema_arrow.metadata or {})
metadata[b'query_embedding'] = json.dumps({
    'format_version': 1,
    'encoder': ENCODER,
    'source_sha256': source_sha256,
    'vector_columns': VECTOR_COLUMNS,
    'query_hash': 'SHA256(UTF8(JSON([query,english,chinese], ensure_ascii=False, separators=(comma,colon))))',
    'runtime': {'torch': torch.__version__, 'transformers': transformers.__version__,
                'inference_dtype': str(inference_dtype), 'device': str(device)},
}, ensure_ascii=False, sort_keys=True).encode('utf-8')
schema = pa.schema([
    *parquet.schema_arrow,
    *[pa.field(name, pa.list_(pa.float32(), ENCODER['dimension'])) for name in VECTOR_COLUMNS.values()],
    pa.field('query_sha256', pa.string()),
], metadata=metadata)

output_path.parent.mkdir(parents=True, exist_ok=True)
with tempfile.NamedTemporaryFile(dir=output_path.parent, suffix='.parquet.partial', delete=False) as handle:
    temporary_path = Path(handle.name)
written = 0
try:
    with pq.ParquetWriter(temporary_path, schema, compression='zstd') as writer:
        for batch in parquet.iter_batches(batch_size=ROW_BATCH_SIZE):
            rows = batch.to_pylist()
            texts = [row[name] for row in rows for name in VECTOR_COLUMNS]
            vectors = encode_texts(texts).reshape(len(rows), 3, ENCODER['dimension'])
            table = pa.Table.from_batches([batch])
            for index, name in enumerate(VECTOR_COLUMNS.values()):
                values = pa.array(vectors[:, index, :].reshape(-1), type=pa.float32())
                table = table.append_column(name, pa.FixedSizeListArray.from_arrays(values, ENCODER['dimension']))
            table = table.append_column('query_sha256', pa.array([query_hash(row) for row in rows]))
            writer.write_table(table.cast(schema))
            written += len(rows)
            print(f'Đã embedding {written:,}/{parquet.metadata.num_rows:,} query', flush=True)
    count = validate_output(temporary_path)
    os.replace(temporary_path, output_path)
finally:
    temporary_path.unlink(missing_ok=True)

print(f'PASS: {count:,} dòng, {count * 3:,} vector × {ENCODER["dimension"]}, float32, L2.')
print(f'Output: {output_path.name} | {output_path.stat().st_size / 1024**2:.2f} MiB')
print(f'Thời gian embedding và ghi file: {perf_counter() - started:.1f} giây')


In [ ]:
if DOWNLOAD_OUTPUT:
    from google.colab import files
    files.download(str(output_path))


## Đọc lại ở hệ thống truy xuất

```python
import json
import pyarrow.parquet as pq

table = pq.read_table('query_rewritten_final_embeddings.parquet')
encoder_info = json.loads(table.schema.metadata[b'query_embedding'])
# Kiểm tra encoder_info['encoder'] với manifest index trước khi sử dụng.
row = table.slice(0, 1).to_pylist()[0]
query_id = row['id']
vector_vi = row['vector_original']
vector_en = row['vector_english']
vector_zh = row['vector_chinese']
```

Pipeline CLI/API nhận trực tiếp file Parquet này và đọc vector có sẵn,
không tự nạp GTE hoặc tính lại embedding. Giữ nguyên các cột và metadata khi chuyển file.
Không gộp trung bình ba vector nếu muốn giữ nguyên cách fusion hiện tại.
